# Study 941 — Short Both Legs — the teardown

The arithmetic-vs-geometric decomposition, the excess-of-cash harvest with its Newey-West *t*, block-bootstrap CIs, the residual-beta regression, the borrow and rebate sweeps, the reset-schedule race, and the live synthetic control that separates cost load from decay. Every real number is frozen from `docs/results.md` (Fingerprint `8f75619ff8e2`), and every harvest number is **before any stock-borrow fee**.

In [1]:
R = {'start': '2010-02-11', 'end': '2026-06-30', 'n_days': 4120, 'fp': '8f75619ff8e2', 'tqqq_mean': 55.48, 'sqqq_mean': -57.25, 'qqq_mean': 20.3, 'bil_mean': 1.32, 'tqqq_vol': 61.2, 'sqqq_vol': 61.3, 'qqq_vol': 20.7, 'implied_load': 2.2, 'sf_long': 2.79, 'sf_short': 1.61, 'sf_share_long': 63, 'decay_3x': 12.9, 'decay_m3x': 25.7, 'mean': 2.06, 'sharpe': 1.64, 't': 8.82, 'vol': 1.26, 'dd': -1.37, 'worst_day': -1.14, 'worst_month': -0.23, 'skew': 1.64, 'kurt': 47.3, 'turnover': 6.8, 'beta': 0.0032, 'alpha': 2.0, 't_alpha': 8.49, 'r2': 0.003, 'qqq_sharpe': 0.92, 'ci_sh_lo': 1.36, 'ci_sh_hi': 1.9, 'ci_m_lo': 1.67, 'ci_m_hi': 2.64, 'ci_frac_neg': 0.0, 'wk_mean': 2.69, 'wk_sharpe': 0.55, 'wk_t': 2.18, 'wk_dd': -11.4, 'wk_wm': -7.1, 'wk_beta': 0.03, 'mo_mean': 5.26, 'mo_sharpe': 0.44, 'mo_t': 2.39, 'mo_dd': -15.5, 'mo_wm': -12.1, 'mo_beta': 0.19, 'mo_alpha': 1.64, 'mo_t_alpha': 0.68, 'nv_mean': -61.15, 'nv_dd': -100.0, 'nv_gross': 51.7, 'nv_date': '2013-08-01', 'be_daily': 2.06, 'be_weekly': 2.7, 'be_monthly': 5.27, 'b1_mean': 1.06, 'b1_t': 4.54, 'b2_mean': 0.06, 'b2_t': 0.26, 'b3_mean': -0.94, 'b3_t': -4.02, 'b5_mean': -2.94, 'b5_t': -12.57, 'b8_mean': -5.94, 'b15_mean': -12.94, 'nr_mean': 0.75, 'nr_sharpe': 0.6, 'nr_t': 3.02, 'nr_be': 0.75, 'nr_early': 1.45, 'nr_early_t': 7.86, 'nr_late': 0.1, 'nr_late_t': 0.22, 'fr_early': 1.5, 'fr_early_t': 7.98, 'fr_late': 2.59, 'fr_late_t': 6.39, 'cost0': 2.2, 'cost0_t': 9.31, 'cost5': 1.86, 'cost10': 1.52, 'yr2020': 6.79, 'yr2022': 3.63, 'yr2016': 0.92, 'yr_all_positive': 17, 'yrs_below_be': 11, 'med': 1.74, 'trim1': 1.82, 'frac_pos': 55.9, 'drop5': 1.77, 'drop5_t': 13.48, 'drop20': 1.4, 'drop20_t': 9.6, 'drop50': 0.95, 'drop50_t': 5.53, 'ex2020': 1.77, 'ex2020_t': 15.97, 'syn_planted': 2.45, 'syn_recovered': 2.45, 'syn_t': 15.06, 'syn_null': 0.0, 'syn_null_t': 0.05, 'syn_decay_l': 14.5, 'syn_decay_s': 29.0}

## 1. The two legs, arithmetically

> 💡 **In plain words:** add up what the two funds returned on an average day, and whatever does not cancel is what shorting them both can pay you.

Annualised arithmetic means (total return, `auto_adjust=True`). Daily-return betas to QQQ are +2.96 / −2.96, so the index cancels almost exactly; what is left is each fund's financing carry and cost load.

⚠️ **This quantity and the section-2 harvest are the same algebra.** On a daily reset with a full rebate, zero borrow and zero cost, `e_nav[t] = r_cash[t] − 0.5·(r_long[t] + r_short[t])` **identically** (agreement to 5e-16). Treat their match as a unit test of the simulator, never as corroboration. And the residual itself is unattributed: expenses, financing spread, the funds' own end-of-day trading, tracking and the BIL-versus-funding basis all sit inside it.

Split by leg, the same residual is lopsided — and the lopsidedness runs against the trade. Each leg pays borrow on its own half of the notional, so each has its own breakeven equal to its own shortfall, and the **crowded, expensive-to-borrow SQQQ leg is the thin one (1.61%/yr)**. The blended 2.06% only exists because the general-collateral TQQQ leg (2.79%/yr, 63% of the harvest) subsidises it.

In [2]:
print(f"TQQQ  mean {R['tqqq_mean']:+.2f}%/yr  vol {R['tqqq_vol']:.1f}%")
print(f"SQQQ  mean {R['sqqq_mean']:+.2f}%/yr  vol {R['sqqq_vol']:.1f}%")
print(f"QQQ   mean {R['qqq_mean']:+.2f}%/yr  vol {R['qqq_vol']:.1f}%")
print(f"BIL   mean {R['bil_mean']:+.2f}%/yr")
print()
print(f"cash - 0.5*(TQQQ + SQQQ) = implied average fund shortfall "
      f"{R['implied_load']:+.2f}%/yr  <- IDENTITY with the harvest in section 2")
print(f"geometric decay on offer  : {R['decay_3x']:.1f}% (3x) + {R['decay_m3x']:.1f}% (-3x) per yr")
print('the decay lives in E[log r], the short book earns E[r] -> uncollectable')
print()
print('split by leg (shortfall vs its own costless replication):')
print(f"  TQQQ {R['sf_long']:+.2f}%/yr ({R['sf_share_long']:.0f}% of the harvest)  "
      f"SQQQ {R['sf_short']:+.2f}%/yr -> the crowded leg is the THIN one")

TQQQ  mean +55.48%/yr  vol 61.2%
SQQQ  mean -57.25%/yr  vol 61.3%
QQQ   mean +20.30%/yr  vol 20.7%
BIL   mean +1.32%/yr

cash - 0.5*(TQQQ + SQQQ) = implied average fund shortfall +2.20%/yr  <- IDENTITY with the harvest in section 2
geometric decay on offer  : 12.9% (3x) + 25.7% (-3x) per yr
the decay lives in E[log r], the short book earns E[r] -> uncollectable

split by leg (shortfall vs its own costless replication):
  TQQQ +2.79%/yr (63% of the harvest)  SQQQ +1.61%/yr -> the crowded leg is the THIN one


## 2. Headline — daily reset, excess-of-cash, zero borrow, 2 bps

In [3]:
print(f"mean {R['mean']:+.2f}%/yr  Sharpe {R['sharpe']:+.2f}  HAC t {R['t']:+.2f}  "
      f"vol {R['vol']:.2f}%  maxDD {R['dd']:.2f}%")
print(f"worst day {R['worst_day']:.2f}%  worst month {R['worst_month']:.2f}%  "
      f"skew {R['skew']:+.2f}  excess kurtosis {R['kurt']:+.1f}  turnover {R['turnover']:.1f}x/yr")
print(f"residual beta to QQQ {R['beta']:+.4f} (R2 {R['r2']:.3f}), "
      f"alpha {R['alpha']:+.2f}%/yr (t={R['t_alpha']:+.2f})")
print(f"QQQ's own excess Sharpe over the window: {R['qqq_sharpe']:+.2f} "
      f"-> the book is neutral to it, not a disguised long")
print()
print(f"bootstrap (2000 draws, 21d blocks): Sharpe 95% CI "
      f"[{R['ci_sh_lo']:+.2f}, {R['ci_sh_hi']:+.2f}], "
      f"mean 95% CI [{R['ci_m_lo']:+.2f}%, {R['ci_m_hi']:+.2f}%], "
      f"frac<0 {R['ci_frac_neg']:.1f}%")

mean +2.06%/yr  Sharpe +1.64  HAC t +8.82  vol 1.26%  maxDD -1.37%
worst day -1.14%  worst month -0.23%  skew +1.64  excess kurtosis +47.3  turnover 6.8x/yr
residual beta to QQQ +0.0032 (R2 0.003), alpha +2.00%/yr (t=+8.49)
QQQ's own excess Sharpe over the window: +0.92 -> the book is neutral to it, not a disguised long

bootstrap (2000 draws, 21d blocks): Sharpe 95% CI [+1.36, +1.90], mean 95% CI [+1.67%, +2.64%], frac<0 0.0%


## 2b. Accrual or a handful of days?

> 💡 **In plain words:** with +47 excess kurtosis, we have to check that the profit is not just the March-2020 week wearing a disguise.

Drop-the-best-days, a trimmed mean, and a straight 2020 exclusion. Note the last line: strip the 50 best days and what survives is *below the breakeven borrow*.

⚠️ **Read the mean column, not the *t* column.** Deleting the best observations and then re-running a *t*-test is not valid inference — it strips the right tail, so the *t* rises **mechanically** (+8.82 → +13.48 after five days) while the mean falls. Those *t*'s are printed as smoothness diagnostics; nothing in the verdict rests on them.

In [4]:
print(f"mean {R['mean']:+.2f}%/yr   median day {R['med']:+.2f}%/yr   "
      f"days > 0: {R['frac_pos']:.1f}%   1% trimmed mean {R['trim1']:+.2f}%/yr")
print(f"drop  5 best days: {R['drop5']:+.2f}%/yr (t {R['drop5_t']:+.2f})")
print(f"drop 20 best days: {R['drop20']:+.2f}%/yr (t {R['drop20_t']:+.2f})")
print(f"drop 50 best days: {R['drop50']:+.2f}%/yr (t {R['drop50_t']:+.2f})  "
      f"<- below the {R['be_daily']:.2f}%/yr breakeven borrow")
print(f"excluding 2020   : {R['ex2020']:+.2f}%/yr (t {R['ex2020_t']:+.2f})")
print()
print(f"and at a borrow equal to the breakeven, {R['yrs_below_be']} of "
      f"{R['yr_all_positive']} calendar years are losers")

mean +2.06%/yr   median day +1.74%/yr   days > 0: 55.9%   1% trimmed mean +1.82%/yr
drop  5 best days: +1.77%/yr (t +13.48)
drop 20 best days: +1.40%/yr (t +9.60)
drop 50 best days: +0.95%/yr (t +5.53)  <- below the 2.06%/yr breakeven borrow
excluding 2020   : +1.77%/yr (t +15.97)

and at a borrow equal to the breakeven, 11 of 17 calendar years are losers


## 3. The reset schedule is a risk dial, not a return dial

> 💡 **In plain words:** letting the position drift looks more profitable, but the extra profit is an accidental bet on the Nasdaq, and the risk is not small.

The equal-dollar short is short gamma on its own rebalance schedule: whichever leg is winning against you grows into the book between resets.

In [5]:
print(f"daily  : mean {R['mean']:+.2f}%  Sharpe {R['sharpe']:+.2f}  t {R['t']:+.2f}  "
      f"DD {R['dd']:.1f}%  worst month {R['worst_month']:.1f}%  beta {R['beta']:+.3f}")
print(f"weekly : mean {R['wk_mean']:+.2f}%  Sharpe {R['wk_sharpe']:+.2f}  t {R['wk_t']:+.2f}  "
      f"DD {R['wk_dd']:.1f}%  worst month {R['wk_wm']:.1f}%  beta {R['wk_beta']:+.3f}")
print(f"monthly: mean {R['mo_mean']:+.2f}%  Sharpe {R['mo_sharpe']:+.2f}  t {R['mo_t']:+.2f}  "
      f"DD {R['mo_dd']:.1f}%  worst month {R['mo_wm']:.1f}%  beta {R['mo_beta']:+.3f}")
print(f"never  : mean {R['nv_mean']:+.2f}%  DD {R['nv_dd']:.1f}%  "
      f"gross short peaked at {R['nv_gross']:.1f}x  -> RUINED {R['nv_date']}")
print()
print(f"monthly's market-neutral alpha is only {R['mo_alpha']:+.2f}%/yr "
      f"(t={R['mo_t_alpha']:+.2f}) -> the headline gain is residual beta, not harvest")

daily  : mean +2.06%  Sharpe +1.64  t +8.82  DD -1.4%  worst month -0.2%  beta +0.003
weekly : mean +2.69%  Sharpe +0.55  t +2.18  DD -11.4%  worst month -7.1%  beta +0.030
monthly: mean +5.26%  Sharpe +0.44  t +2.39  DD -15.5%  worst month -12.1%  beta +0.190
never  : mean -61.15%  DD -100.0%  gross short peaked at 51.7x  -> RUINED 2013-08-01

monthly's market-neutral alpha is only +1.64%/yr (t=+0.68) -> the headline gain is residual beta, not harvest


## 4. The crux — borrow, an ASSUMPTION swept end to end

> 💡 **In plain words:** the fee for borrowing the shares is the same size as the profit, so the whole verdict hangs on a number we cannot observe for free.

Borrow is charged on the gross short notional (1.0x NAV), so the harvest falls one-for-one with the assumed rate.

In [6]:
print(f"borrow  0%: mean {R['mean']:+.2f}%/yr  (t {R['t']:+.2f})")
print(f"borrow  1%: mean {R['b1_mean']:+.2f}%/yr  (t {R['b1_t']:+.2f})")
print(f"borrow  2%: mean {R['b2_mean']:+.2f}%/yr  (t {R['b2_t']:+.2f})")
print(f"borrow  3%: mean {R['b3_mean']:+.2f}%/yr  (t {R['b3_t']:+.2f})")
print(f"borrow  5%: mean {R['b5_mean']:+.2f}%/yr  (t {R['b5_t']:+.2f})")
print(f"borrow 15%: mean {R['b15_mean']:+.2f}%/yr")
print()
print(f"breakeven blended borrow: daily {R['be_daily']:.2f}%/yr, "
      f"weekly {R['be_weekly']:.2f}%/yr, monthly {R['be_monthly']:.2f}%/yr")
print('(the weekly/monthly headroom is bought with the tail in section 3)')

borrow  0%: mean +2.06%/yr  (t +8.82)
borrow  1%: mean +1.06%/yr  (t +4.54)
borrow  2%: mean +0.06%/yr  (t +0.26)
borrow  3%: mean -0.94%/yr  (t -4.02)
borrow  5%: mean -2.94%/yr  (t -12.57)
borrow 15%: mean -12.94%/yr

breakeven blended borrow: daily 2.06%/yr, weekly 2.70%/yr, monthly 5.27%/yr
(the weekly/monthly headroom is bought with the tail in section 3)


## 5. Account terms — the rebate is a precondition, not a source

The two legs' own financing carry averages to **+1x cash on the liability side**; the interest on the short proceeds is exactly what offsets it. So a full-rebate account's excess return is the cost residual in full, and a no-rebate account *forfeits the cash rate itself* (1.32%/yr here) — a cost of the account terms, not a component of the alpha. It happens to be worth more than half the harvest, and it grows with the level of rates.

In [7]:
print(f"full rebate: mean {R['mean']:+.2f}%/yr  Sharpe {R['sharpe']:+.2f}  t {R['t']:+.2f}  "
      f"breakeven borrow {R['be_daily']:.2f}%/yr")
print(f"   eras: 2010-2017 {R['fr_early']:+.2f}% (t={R['fr_early_t']:+.2f})  |  "
      f"2018-2026 {R['fr_late']:+.2f}% (t={R['fr_late_t']:+.2f})")
print(f"no rebate  : mean {R['nr_mean']:+.2f}%/yr  Sharpe {R['nr_sharpe']:+.2f}  t {R['nr_t']:+.2f}  "
      f"breakeven borrow {R['nr_be']:.2f}%/yr")
print(f"   eras: 2010-2017 {R['nr_early']:+.2f}% (t={R['nr_early_t']:+.2f})  |  "
      f"2018-2026 {R['nr_late']:+.2f}% (t={R['nr_late_t']:+.2f})  <- dead")
print()
print(f"cost sweep (one-way bps): 0 -> {R['cost0']:+.2f}% (t {R['cost0_t']:+.2f}), "
      f"5 -> {R['cost5']:+.2f}%, 10 -> {R['cost10']:+.2f}%  (turnover {R['turnover']:.1f}x/yr)")

full rebate: mean +2.06%/yr  Sharpe +1.64  t +8.82  breakeven borrow 2.06%/yr
   eras: 2010-2017 +1.50% (t=+7.98)  |  2018-2026 +2.59% (t=+6.39)
no rebate  : mean +0.75%/yr  Sharpe +0.60  t +3.02  breakeven borrow 0.75%/yr
   eras: 2010-2017 +1.45% (t=+7.86)  |  2018-2026 +0.10% (t=+0.22)  <- dead

cost sweep (one-way bps): 0 -> +2.20% (t +9.31), 5 -> +1.86%, 10 -> +1.52%  (turnover 6.8x/yr)


## 6. Live synthetic control — fee load vs decay, by construction

> 💡 **In plain words:** two toy markets, identical melt in the levered funds, one with fees and one without. Only the fee shows up in the harvest.

Both worlds carry the same large `0.5*L*(L-1)*σ²` decay; `signal_strength` scales only the funds' expense-plus-financing load. Six seeds each.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from short_pair import data, strategy as st
for tag, ss in [('costly funds (planted)', 1.0), ('free funds (null)     ', 0.0)]:
    means, ts = [], []
    for px, truth in data.synthetic_panel(n_seeds=6, signal_strength=ss):
        d = st.synthetic_detect(px, cost_bps=0.0)
        means.append(d['ann_mean']); ts.append(d['t_hac'])
    print('%s planted %+.2f%%/yr -> recovered %+.2f%%/yr (sd %.2f%%), max|t| %.2f, '
          'decay in legs %.1f%%/%.1f%% per yr'
          % (tag, truth['expected_harvest_ann']*100, np.mean(means)*100,
             np.std(means, ddof=1)*100, max(abs(t) for t in ts),
             truth['decay_long_ann']*100, truth['decay_short_ann']*100))

costly funds (planted) planted +2.45%/yr -> recovered +2.45%/yr (sd 0.00%), max|t| 15.06, decay in legs 14.5%/29.0% per yr


free funds (null)      planted +0.00%/yr -> recovered +0.00%/yr (sd 0.00%), max|t| 0.05, decay in legs 14.5%/29.0% per yr


## Verdict

- **Signal — Real.** Excess-of-cash **+2.06%/yr before borrow**, HAC *t* = **+8.82**, bootstrap mean CI [+1.67%, +2.64%] clear of zero, residual beta +0.0032 to QQQ with alpha *t* = +8.49, positive in every calendar year, in both eras (+1.50% / +2.59%), and holding up as a magnitude without its 50 best days (+0.95%) or without 2020 (+1.77%) — no *t* quoted for those two, per §2b. What is *not* evidence: the harvest equalling the implied load (+2.20%/yr) is an **identity**, and the load is an **unattributed residual**. What is evidence: the synthetic control recovers a planted load (+2.45% → +2.45%, max|*t*| 15.1) while returning +0.00% (max|*t*| 0.05) when the same 14%/29% decay carries no fee — so the volatility decay is *not* what is being collected. **Pair selection:** the universe is n = 1 and hindsight-chosen. For the *signal* that is conservative (TQQQ/SQQQ is the cheapest, best-run levered pair, so its residual is near a floor); for the *borrow* it runs the other way, and a fund being wound up mid-sample would **not** have paid a short — an ETF liquidation pays NAV and the short closes flat.
- **Tradability — Fragile.** Breakeven blended borrow **2.06%/yr**, but split by leg the cheap TQQQ side carries 63% of the harvest (2.79%/yr) and the crowded SQQQ side breaks even at only **1.61%/yr**; at the blended rate 11 of 17 calendar years lose money; no rebate on the proceeds cuts the harvest to +0.75%/yr and to +0.10%/yr (*t* = +0.22) since 2018 — a Mirage at retail terms. The apparent fix — reset less often — buys residual beta, not alpha (+1.64%/yr, *t* = +0.68), at Sharpe +0.44 and a -12.1% month; not resetting at all ruins the book on 2013-08-01 at 51.7x gross. It is a prime-brokerage carry of ~2%/yr on 1.3% vol, entirely at the mercy of a fee that exists to price it away.